Re-attach (Always Run First)
This restores all variables after a cluster reset. Run it every time you open your notebook.

In [0]:
# Re-attach cell — run this first every session
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)
from pyspark.sql.functions import col

CATALOG    = 'ecommerce'
MODEL_PATH = '/Volumes/ecommerce/sc_ecommerce/vol_ecommerce/models/'
EXP_NAME   = '/ecom_purchase_prediction_2'
MODEL_NAME = 'ecommerce.default.ecom_purchase_model'  # full UC path: catalog.schema.model
LABEL_COL  = 'will_purchase'

mlflow.set_experiment(EXP_NAME)

# Reload data from Delta (always safe after reset)
df_all   = spark.table(f'{CATALOG}.gold.ml_dataset')
df_train = df_all.filter(col('split') == 'train')
df_test  = df_all.filter(col('split') == 'test')

# Reload evaluators
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='will_purchase',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='will_purchase',
    predictionCol='prediction',
    metricName='f1'
)

print('Re-attached!')
print(f'Train: {df_train.count():,}  |  Test: {df_test.count():,}')


Re-attached!
Train: 4,579,212  |  Test: 1,142,381


### View All Logged Runs
Pull every run from your experiment into a table. This shows you everything you trained in Day 6 — ranked by AUC so the best model is always at the top.

In [0]:
# Pull all runs from the experiment and display them
# mlflow.search_runs() returns a pandas DataFrame of all your runs
runs = mlflow.search_runs(
    experiment_names = [EXP_NAME],
    order_by         = ['metrics.test_auc DESC']   # best AUC first
)

# Select only the columns we care about
cols = ['tags.mlflow.runName', 'metrics.test_auc', 'metrics.test_f1',
        'params.model', 'params.numTrees', 'params.maxDepth',
        'params.regParam', 'run_id', 'start_time']

available_cols = [c for c in cols if c in runs.columns]
display(spark.createDataFrame(runs[available_cols]))

print(f'Total runs logged so far: {len(runs)}')


tags.mlflow.runName,metrics.test_auc,metrics.test_f1,params.model,params.numTrees,params.maxDepth,params.regParam,run_id,start_time
rf_trees100_depth10,0.9656682210092149,0.9064032179964278,RandomForest,100,10,null,4d4b0b439e464daaa25c637010ec5c09,2026-03-03T07:23:39.800Z
rf_trees20_depth10,0.9647206195479434,0.9025137306541979,RandomForest,20,10,null,84cbe55b1ee140a7be280090dd130168,2026-03-03T07:19:21.997Z
rf_trees20_depth5,0.9443894868205983,0.8803771083119662,RandomForest,20,5,null,b44d34aaad784bd48a895f331dfc230d,2026-03-03T07:18:43.169Z
rf_trees100_depth5,0.9441040503996893,0.8820823754393283,RandomForest,100,5,null,19992d9e07da4a7788850f5b83475877,2026-03-03T07:20:35.647Z
random_forest,0.9438333842000194,0.8828638899374222,RandomForest,50,5,null,b89aec3e2e9d4d12a4bbd8e484400432,2026-03-03T07:17:06.952Z
logistic_regression,0.9124694272603999,0.8900675010733182,LogisticRegression,null,null,0.01,78581e312490463fb6fe4f5a5ef18158,2026-03-03T07:16:17.679Z


Total runs logged so far: 6


In [0]:
# Find the Best Run
# Programmatically identify which run has the highest AUC. This is the run whose model we will register.

#  Find the best run by AUC

# Drop rows with no test_auc (e.g. the connectivity test run from Day 6)
runs_with_auc = runs.dropna(subset=['metrics.test_auc'])

# First row = highest AUC (already sorted)
best_run      = runs_with_auc.iloc[0]
best_run_id   = best_run['run_id']
best_run_name = best_run['tags.mlflow.runName']
best_auc      = best_run['metrics.test_auc']
best_f1       = best_run['metrics.test_f1']

print('=' * 50)
print('BEST RUN')
print('=' * 50)
print(f'Run name : {best_run_name}')
print(f'AUC      : {best_auc:.4f}')
print(f'F1       : {best_f1:.4f}')
print(f'Run ID   : {best_run_id}')
print()
print('This model will be registered in the next step.')


BEST RUN
Run name : rf_trees100_depth10
AUC      : 0.9657
F1       : 0.9064
Run ID   : 4d4b0b439e464daaa25c637010ec5c09

This model will be registered in the next step.


In [0]:
# : Print a Clean Comparison Table
# Print all your runs in a readable side-by-side table so you can see at a glance how each model performed. This is what you would share in a team meeting or learning post.

# Cell 4 — Clean comparison table in terminal output
print('=' * 60)
print('ALL MODEL RUNS — RANKED BY AUC')
print('=' * 60)
print(f'{"Run Name":<35} {"AUC":>8} {"F1":>8}')
print('-' * 60)

for _, row in runs_with_auc.iterrows():
    name   = str(row.get('tags.mlflow.runName', 'unnamed'))[:34]
    auc    = row.get('metrics.test_auc', 0)
    f1     = row.get('metrics.test_f1',  0)
    marker = '  <- BEST' if row['run_id'] == best_run_id else ''
    print(f'{name:<35} {auc:>8.4f} {f1:>8.4f}{marker}')

print('=' * 60)


ALL MODEL RUNS — RANKED BY AUC
Run Name                                 AUC       F1
------------------------------------------------------------
rf_trees100_depth10                   0.9657   0.9064  <- BEST
rf_trees20_depth10                    0.9647   0.9025
rf_trees20_depth5                     0.9444   0.8804
rf_trees100_depth5                    0.9441   0.8821
random_forest                         0.9438   0.8829
logistic_regression                   0.9125   0.8901


### Add a Tag to the Best Run
Tags are sticky notes on a run. You can add them at any time — even after training is done. They help you filter runs in the UI and leave notes about why a run matters.

In [0]:
#  Tag the best run so it stands out in the UI
# You can add tags to any existing run using its run_id
with mlflow.start_run(run_id=best_run_id):
    mlflow.set_tag('status',   'best_model')
    mlflow.set_tag('dataset',  'ecommerce_oct_nov_2019')
    mlflow.set_tag('promoted', 'day7')

print(f'Tags added to run: {best_run_name}')
print('In the UI: click the run -> Tags tab to see them')


Tags added to run: rf_trees100_depth10
In the UI: click the run -> Tags tab to see them


### Register the Best Model
Registering the model gives it an official name and version number in the MLflow Model Registry. This is the step that takes a model from an experiment and makes it official — ready to be used in production.

In [0]:
# Register the best model in the MLflow Model Registry
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Step 1: Create the model name in the registry (if not already there)
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = 'Predicts whether an ecommerce user will make a purchase.'
    )
    print(f'Created model: {MODEL_NAME}')
except Exception:
    print(f'Model already exists: {MODEL_NAME} — adding new version')


Model already exists: ecom_purchase_model — adding new version


In [0]:
# ── FIX: Unity Catalog requires a model signature ────────────────────────
# A signature tells MLflow exactly what columns the model expects as input
# and what it returns as output. Unity Catalog enforces this — no signature
# = registration rejected. We infer it automatically from a small sample.
# ─────────────────────────────────────────────────────────────────────────

from pyspark.ml import PipelineModel
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.ml.functions import vector_to_array

TMP_DIR = '/Volumes/ecommerce/sc_ecommerce/vol_ecommerce/mlflow_tmp'

# Step 1: Load the best model
best_pipeline = PipelineModel.load(f'{MODEL_PATH}rf_best_tuned')
print('Model loaded from Volume')

# Step 2: Create a small sample to infer signature from
# Signature = input column names+types and output column names+types
# infer_signature() figures this out automatically from a pandas sample
FEATURE_COLS = [
    'total_events', 'total_sessions', 'total_views',
    'total_cart_adds', 'avg_price_viewed'
]

sample_spark  = df_test.select(FEATURE_COLS).limit(100)
sample_pandas = sample_spark.toPandas()   # convert small sample to pandas

# Run model on sample to get output shape
sample_preds  = best_pipeline.transform(sample_spark)

# Extract prediction and probability as plain numbers (not vectors)
from pyspark.ml.functions import vector_to_array
sample_output = (
    sample_preds
    .withColumn('buy_probability', vector_to_array('probability')[1])
    .select('prediction', 'buy_probability')
    .toPandas()
)

# Infer signature from input pandas df and output pandas df
signature = infer_signature(
    model_input  = sample_pandas,   # input: 5 feature columns
    model_output = sample_output    # output: prediction + probability
)
print('Signature inferred:')
print(signature)

# Step 3: Log and register with signature + input_example
with mlflow.start_run(run_name='register_best_model') as reg_run:
    mlflow.log_param('model',    'RandomForest')
    mlflow.log_param('numTrees', 100)
    mlflow.log_param('maxDepth', 10)
    mlflow.log_metric('test_auc', best_auc)
    mlflow.log_metric('test_f1',  best_f1)

    mlflow.spark.log_model(
        spark_model           = best_pipeline,
        artifact_path         = 'model',
        dfs_tmpdir            = TMP_DIR,
        signature             = signature,       # <-- fixes Unity Catalog error
        input_example         = sample_pandas,  # <-- shows example input
        registered_model_name = MODEL_NAME
    )
    reg_run_id = reg_run.info.run_id

print(f'\nRegistered: {MODEL_NAME}')
print(f'Run ID:     {reg_run_id}')
print(f'Find it:    Left sidebar -> Models -> {MODEL_NAME}')


In [0]:
# Add a description to the newly registered version
# Tells anyone who finds the model exactly what it is
import time

client = MlflowClient()

# Get the latest version number
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in versions])
print(f'Latest version: {latest_version}')

client.update_model_version(
    name        = MODEL_NAME,
    version     = str(latest_version),
    description = (
        f'Random Forest. AUC={best_auc:.4f}. F1={best_f1:.4f}. '
        'Trained on ecommerce Oct/Nov 2019. 5.7M users. '
        'numTrees=100, maxDepth=10, seed=42. '
        'Features: total_events, sessions, views, cart_adds, avg_price_viewed. '
        'Class weights applied: 3.3x for purchasers.'
    )
)
print(f'Description added to {MODEL_NAME} v{latest_version}')


In [0]:
# Load the registered model back from MLflow and verify it works
# This is the exact same pattern Day 8 will use for batch inference

# Load from Unity Catalog Model Registry by name + version
model_uri = f'models:/{MODEL_NAME}/{latest_version}'
print(f'Loading from: {model_uri}')

loaded_model = mlflow.spark.load_model(model_uri)
print('Model loaded from registry!')

# Run on a small sample to verify it predicts correctly
sample_preds = loaded_model.transform(df_test.limit(500))
sample_auc   = auc_evaluator.evaluate(sample_preds)

print(f'\nVerification AUC (500 rows): {sample_auc:.4f}')
print(f'Full test AUC (Day 6):       {best_auc:.4f}')
print()
if abs(sample_auc - best_auc) < 0.05:
    print('Registered model is working correctly — ready for Day 8!')
else:
    print('AUC gap > 0.05 — double check the model path or version number')

display(sample_preds.select(
    'will_purchase', 'prediction', 'probability',
    'total_events', 'total_cart_adds'
).limit(10))


## Where to Find Your Model — Complete Guide

### In the Databricks UI
1. **Left sidebar** → click the **Models** icon (looks like a box with a tag)
2. Click **`ecom_purchase_model`** — you will see all versions listed
3. Click **Version 1** (or your latest version) to open the version detail page
4. On the version page you can see:
   - **Description** — what you added above
   - **Source Run** — a clickable link back to the exact Day 6 training run
   - **Schema** — input features the model expects
   - **Metrics** — AUC and F1 logged during registration

### In Code — Three Ways to Load
```python
# Option 1: Load specific version (most common for production)
model = mlflow.spark.load_model('models:/ecom_purchase_model/1')

# Option 2: Load latest version
model = mlflow.spark.load_model('models:/ecom_purchase_model/latest')

# Option 3: Load from Volume path directly (fallback)
from pyspark.ml import PipelineModel
model = PipelineModel.load('/Volumes/ecommerce/sc_ecommerce/vol_ecommerce/models/rf_best_tuned')
```

### Why the Model Registry Is Useful
| Without Registry | With Registry |
|---|---|
| Model is just a folder on disk — easy to forget | Has a name, version, description — easy to find |
| No link to the training run | Clickable link → exact run → exact params and metrics |
| Anyone needs to know the Volume path | Anyone loads it with `models:/ecom_purchase_model/1` |
| No version history | v1 → v2 → v3 all kept, can roll back anytime |
| Cannot tell if it is in use | Can be tagged as Staging or Production |

### What Each Field Means on the Version Page
- **Status: READY** — model is available to load and use
- **Source Run** — the Day 6 training run that produced this model
- **Registered At** — timestamp when you registered it
- **Schema (inputs)** — the 5 feature columns the model expects
- **Tags** — any labels you added (status: best_model etc.)


In [0]:
# View All Model Versions
# Check what is registered. Every time you train a better model and register it, it gets the next version number. This gives you a full history of every model you ever promoted.

# Cell 8 — List all versions of the registered model
client = MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")

print(f'=== REGISTERED VERSIONS: {MODEL_NAME} ===')
for v in versions:
    print(f'  Version  : {v.version}')
    print(f'  Status   : {v.status}')
    print(f'  Run ID   : {v.run_id}')
    desc = (v.description or '(no description)')[:80]
    print(f'  Desc     : {desc}')
    print()
